In [ ]:
# General notebook settings
import logging

import pypsa

logging.getLogger("gurobipy").propagate = False
pypsa.options.params.optimize.log_to_console = False

# Data centre investment with a willingness to pay

A data centre is a flexible consumer. It decides how much capacity to build and runs whenever electricity is cheaper than the value of its computation. We add one to the single-node model in the style of [model.energy](https://model.energy).

In [ ]:
import matplotlib.pyplot as plt

import pypsa

n = pypsa.examples.model_energy()

We model the data centre as a `Generator` with `sign=-1`. Its dispatch $g_{n,s,t} \geq 0$ enters the nodal balance as withdrawal $-g_{n,s,t}$. The objective ignores `sign`, so the utility of computation enters as a negative marginal cost $o_{n,s,t} < 0$:

$$\min \; \ldots + c_{n,s} G_{n,s} + \sum_t w_t^o \left( o_{n,s,t} \, g_{n,s,t} + qmc_{n,s,t} \, g_{n,s,t}^2 \right)$$

The data centre stands for a fleet in aggregate. Its marginal willingness to pay $-o_{n,s,t} - 2 \, qmc_{n,s,t} \, g_{n,s,t}$ falls from 500 €/MWh to zero at 2 GW, because workloads differ in value. The capital cost $c_{n,s}$ annualises about 10 M€/MW for building, cooling and grid connection over 20 years, excluding servers. All numbers are illustrative.

In [ ]:
c = 1e6  # €/MW/a
o = -500  # €/MWh
qmc = 0.125  # €/MW²h

n.add("Carrier", "data centre")
n.add(
    "Generator",
    "data centre",
    bus="electricity",
    carrier="data centre",
    sign=-1,
    p_nom_extendable=True,
    capital_cost=c,
    marginal_cost=o,
    marginal_cost_quadratic=qmc,
)
n.optimize(solver_name="gurobi")

The model invests in the data centre alongside wind, solar and storage. Statistics report its consumption as withdrawal.

In [ ]:
n.statistics(components=["Generator", "Load"])[
    ["Optimal Capacity", "Capacity Factor", "Withdrawal"]
]

Below capacity, the data centre consumes until its marginal willingness to pay equals the electricity price. At capacity, the price falls below it.

In [ ]:
p = n.generators_t.p["data centre"] / 1e3
price = n.buses_t.marginal_price["electricity"]

fig, ax = plt.subplots(layout="constrained")
ax.scatter(p, price, s=4, alpha=0.3, color="#2993b5")
ax.plot([0, -o / (2 * qmc) / 1e3], [-o, 0], color="#D10A49")
ax.axvline(p.max(), color="grey", ls=":")
ax.text(1.4, 200, "marginal willingness to pay", color="#D10A49")
ax.text(p.max() + 0.02, 560, "capacity", color="grey")
ax.set(
    xlabel="data centre consumption (GW)",
    ylabel="electricity price (€/MWh)",
    ylim=(0, 600),
)